## 💻 Ambiente de Execução e Avaliação (Inferência)

O pipeline de inferência multibase para a consolidação do estudo de ablação foi executado em ambiente de nuvem, garantindo a reprodutibilidade dos cálculos de métricas forenses em hardwares com restrição de memória.

**Especificações da Infraestrutura:**
* **Plataforma:** Kaggle Notebooks
* **Acelerador de Hardware:** NVIDIA Tesla T4 GPU (16GB VRAM)
* **Framework Base:** PyTorch, Pandas e `scikit-learn`
* **Processamento:** CUDA (`cuda:0`) ativado para aceleração do fluxo de tensores durante as predições.

**Resumo da Matriz de Avaliação (Modelos vs. Bases):**
* **Variantes do Classificador Híbrido:** `1_completo`, `2_sem_prompt`, `3_sem_lbp`
* **Bases Independentes de Teste:** AIGI-Holmes, Artifact, SID-Set e Cozzolino-Test-Set

**Resultados Finais do Estudo de Ablação:**
Os pesos extraídos de cada variante foram avaliados sob o mesmo limiar analítico. Os resultados comprovam a superioridade da estratégia de *Early Fusion* (1_completo) na generalização contra dados *Out-of-Distribution* (OOD).

| Dataset de Teste | Variante do Modelo | Acurácia | AUC-ROC |
| :--- | :--- | :---: | :---: |
| **AIGI-Holmes-Test** | **1_completo (Proposto)** | **92.66%** | **0.9849** |
| | 2_sem_prompt | 92.43% | 0.9844 |
| | 3_sem_lbp | 91.17% | 0.9823 |
| **Artifact-Test** | **1_completo (Proposto)** | **82.95%** | **0.9117** |
| | 2_sem_prompt | 81.75% | 0.9063 |
| | 3_sem_lbp | 81.47% | 0.9079 |
| **Cozzolino-Test-Set** | **1_completo (Proposto)** | **83.28%** | **0.9560** |
| | 2_sem_prompt | 80.19% | 0.9531 |
| | 3_sem_lbp | 75.01% | 0.9486 |
| **SID-Set-Test** | 1_completo (Proposto) | 98.35% | **0.9991** |
| | 2_sem_prompt | 98.55% | 0.9990 |
| | 3_sem_lbp | **98.72%** | 0.9990 |
| **MÉDIA GERAL (OVERALL)** | **1_completo (Proposto)** | **89.31%** | **0.9629** |
| | 2_sem_prompt | 88.23% | 0.9607 |
| | 3_sem_lbp | 86.59% | 0.9595 |

## **Importações, Clone do Repositório e Configurações Iniciais**

In [1]:
import os
import sys
import gc
import glob
import random
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from sklearn.metrics import accuracy_score, roc_auc_score

# Configuração de cache para evitar poluição no diretório de trabalho
os.environ["HF_DATASETS_CACHE"] = "/kaggle/working/datasets_cache"

# Clone do repositório para puxar a arquitetura oficial do CLIPImageClassifier
!git clone https://github.com/Jvfrostbr/ClipBased-SyntheticImageDetection.git
!pip install open_clip_torch scikit-learn

# Inserção da pasta networks no path do sistema
%cd /kaggle/working/ClipBased-SyntheticImageDetection
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from networks.clip_custom_detector import CLIPImageClassifier

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Ambiente de Inferência Configurado! Dispositivo: {DEVICE}")

class ForensicTestDataset(Dataset):
    def __init__(self, samples, preprocess_fn):
        self.samples = samples
        self.preprocess_fn = preprocess_fn

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        amostra = self.samples[idx]
        try:
            image = Image.open(amostra['file_path']).convert('RGB')
            pixel_values = self.preprocess_fn(image)
        except Exception:
            # Fallback robusto para evitar travamento em imagens corrompidas no Kaggle
            pixel_values = torch.zeros((3, 224, 224))
            
        return pixel_values, amostra['label']

Cloning into 'ClipBased-SyntheticImageDetection'...
remote: Enumerating objects: 154, done.
remote: Counting objects: 100% (96/96), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 154 (delta 48), reused 68 (delta 35), pack-reused 58 (from 1)
Receiving objects: 100% (154/154), 616.88 KiB | 12.85 MiB/s, done.
Resolving deltas: 100% (59/59), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.2 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3

## **Mapeamento dos Datasets de Teste**

In [2]:
PATHS_TEST = {
    "AIGI-Holmes-Test": "/kaggle/input/datasets/jvfrostbr/aigi-holmes-test",
    "Artifact-Test": "/kaggle/input/datasets/jvfrostbr/datasets-test/dataset_inferencia_estatico/artifact",
    "SID-Set-Test": "/kaggle/input/datasets/jvfrostbr/datasets-test/dataset_inferencia_estatico/sidset",
    "Cozzolino-Test-Set": "/kaggle/input/datasets/jvfrostbr/datasets-test/test_set_cozolino/test_set_cozolino/test_set"
}

datasets_mapeados = {}

print("📊 A iniciar o varrimento e mapeamento flexível dos conjuntos de teste...")

for nome_base, root_path in PATHS_TEST.items():
    amostras = []
    
    # Lógica de mapeamento exclusiva para a estrutura de pastas do Cozolino et al.
    if nome_base == "Cozzolino-Test-Set":
        pastas_modelos = glob.glob(os.path.join(root_path, "*"))
        for pasta in pastas_modelos:
            if os.path.isdir(pasta):
                nome_pasta = os.path.basename(pasta).lower()
                
                # Regra: Pastas que começam com "real_" recebem label 0. O resto recebe 1.
                label = 0 if nome_pasta.startswith("real_") else 1
                
                imagens = glob.glob(os.path.join(pasta, "*.*"))
                for img_path in imagens:
                    if img_path.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                        amostras.append({'file_path': img_path, 'label': label})
                        
    # Lógica padrão para AIGI-Holmes, Artifact e SID-Set
    else:
        dir_real = os.path.join(root_path, "**", "0_real", "**", "*.*")
        dir_fake = os.path.join(root_path, "**", "1_fake", "**", "*.*")
        
        for path in glob.glob(dir_real, recursive=True):
            if path.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                amostras.append({'file_path': path, 'label': 0})
                
        for path in glob.glob(dir_fake, recursive=True):
            if path.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                amostras.append({'file_path': path, 'label': 1})
            
    random.shuffle(amostras)
    datasets_mapeados[nome_base] = amostras
    print(f"✅ {nome_base}: {len(amostras)} imagens localizadas.")

print("\n🚀 Todos os datasets de teste foram integrados com sucesso!")

📊 A iniciar o varrimento e mapeamento flexível dos conjuntos de teste...
✅ AIGI-Holmes-Test: 10000 imagens localizadas.
✅ Artifact-Test: 10000 imagens localizadas.
✅ SID-Set-Test: 10000 imagens localizadas.
✅ Cozzolino-Test-Set: 24000 imagens localizadas.

🚀 Todos os datasets de teste foram integrados com sucesso!


## **Definindo Função de Inferência e Avaliação**

In [3]:
def avaliar_modelo(config, samples, preprocess_fn, batch_size=64):
    """
    Instancia a variação do classificador correspondente, carrega os pesos do 
    checkpoint e roda a inferência calculando métricas robustas.
    """
    # Garante que a arquitetura nasça com as flags corretas da variante avaliada
    model = CLIPImageClassifier(
        use_prompt=config['prompt'],
        use_lbp=config['lbp'],
        multiclass=config['multiclass']
    ).to(DEVICE)
    
    print(f"💾 Carregando pesos estruturados de: {config['checkpoint_path']}")
    checkpoint = torch.load(config['checkpoint_path'], map_location=DEVICE)
    
    # Compatibilidade dupla para state_dict puro ou empacotado pelo pipeline de treino
    if 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint)
        
    model.eval()
    
    test_dataset = ForensicTestDataset(samples, preprocess_fn)
    test_loader = DataLoader(
        test_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=4, 
        pin_memory=True
    )
    
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc=f"Inferência ({config['nome']})"):
            images = images.to(DEVICE)
            outputs = model(images)
            
            # Aplicando um  Sigmoid para extrair a probabilidade
            # contínua da imagem pertencer à classe sintética (1_fake)
            probs = torch.sigmoid(outputs).cpu().numpy()
            
            all_preds.extend(probs)
            all_targets.extend(labels.numpy())
            
    all_preds = np.array(all_preds).squeeze()
    all_targets = np.array(all_targets)
    
    # Threshold padrão de corte em 0.50 para cálculo de acurácia discreta
    all_preds_bin = (all_preds > 0.5).astype(int)
    
    # Extração das métricas oficiais via Sklearn
    acc = accuracy_score(all_targets, all_preds_bin)
    auc = roc_auc_score(all_targets, all_preds)
    
    del model
    gc.collect()
    torch.cuda.empty_cache()
    
    return acc, auc

## **Loop de Execução do Estudo de Ablação:**

In [4]:
# Mapeamento detalhado ligando as configurações aos caminhos dos checkpoints gerados
experimentos_teste = {
    "v1": {
        "nome": "1_completo",
        "prompt": True,
        "lbp": True,
        "multiclass": False,
        "checkpoint_path": "/kaggle/input/datasets/jvfrostbr/pesos-ep-1/checkpoint_ep1_v1.pth"
    },
    "v2": {
        "nome": "2_sem_prompt",
        "prompt": False,
        "lbp": True,
        "multiclass": False,
        "checkpoint_path": "/kaggle/input/datasets/jvfrostbr/pesos-ep-1/checkpoint_ep1_v2.pth"
    },
    "v3": {
        "nome": "3_sem_lbp",
        "prompt": True,
        "lbp": False,
        "multiclass": False,
        "checkpoint_path": "/kaggle/input/datasets/jvfrostbr/pesos-ep-1/checkpoint_ep1_v3.pth"
    }
}

resultados_globais = []

# Instanciação temporária estritamente para extrair o operador de preprocessamento nativo do ViT-L-14
model_temp = CLIPImageClassifier(use_prompt=False, use_lbp=False, multiclass=False)
preprocess_fn = model_temp.preprocess
del model_temp
gc.collect()

# Execução da matriz cruzada: 3 Modelos x 3 Bases de Teste
for chave, config in experimentos_teste.items():
    print("\n" + "="*70)
    print(f"🚀 AVALIANDO ARCH: {config['nome'].upper()}")
    print("="*70)
    
    for nome_dataset, amostras in datasets_mapeados.items():
        if len(amostras) == 0:
            print(f"⚠️ Base {nome_dataset} não possui amostras. Pulando...")
            continue
            
        acc, auc = avaliar_modelo(config, amostras, preprocess_fn, batch_size=64)
        
        resultados_globais.append({
            "Variante": config['nome'],
            "Dataset Teste": nome_dataset,
            "Acurácia": f"{acc*100:.2f}%",
            "AUC-ROC": f"{auc:.4f}"
        })

# Construção e plotagem da tabela final consolidada
df_resultados = pd.DataFrame(resultados_globais)
print("\n" + "="*60)
print("🏁 ESTUDO DE ABLAÇÃO FINALIZADO - RESULTADOS DE INFERÊNCIA")
print("="*60)
df_resultados.sort_values(by=["Dataset Teste", "Variante"], inplace=True)
display(df_resultados)

open_clip_model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(



🚀 AVALIANDO ARCH: 1_COMPLETO
💾 Carregando pesos estruturados de: /kaggle/input/datasets/jvfrostbr/pesos-ep-1/checkpoint_ep1_v1.pth


Inferência (1_completo): 100%|██████████| 157/157 [09:59<00:00,  3.82s/it]
/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


💾 Carregando pesos estruturados de: /kaggle/input/datasets/jvfrostbr/pesos-ep-1/checkpoint_ep1_v1.pth


...

Inferência (3_sem_lbp): 100%|██████████| 157/157 [08:38<00:00,  3.30s/it]
/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


💾 Carregando pesos estruturados de: /kaggle/input/datasets/jvfrostbr/pesos-ep-1/checkpoint_ep1_v3.pth


Inferência (3_sem_lbp): 100%|██████████| 375/375 [20:38<00:00,  3.30s/it]



🏁 ESTUDO DE ABLAÇÃO FINALIZADO - RESULTADOS DE INFERÊNCIA


,Variante,Dataset Teste,Acurácia,AUC-ROC
0,1_completo,AIGI-Holmes-Test,92.66%,0.9849
4,2_sem_prompt,AIGI-Holmes-Test,92.43%,0.9844
8,3_sem_lbp,AIGI-Holmes-Test,91.17%,0.9823
1,1_completo,Artifact-Test,82.95%,0.9117
5,2_sem_prompt,Artifact-Test,81.75%,0.9063
9,3_sem_lbp,Artifact-Test,81.47%,0.9079
3,1_completo,Cozzolino-Test-Set,83.28%,0.9560
7,2_sem_prompt,Cozzolino-Test-Set,80.19%,0.9531
11,3_sem_lbp,Cozzolino-Test-Set,75.01%,0.9486
2,1_completo,SID-Set-Test,98.35%,0.9991
